In [ ]:
#Qué hace: construye variables de laboratorio diarias, manejando irregularidad y ausencia de medidas, y las organiza por dominios.
#Clave: ausencia de lab ≠ normalidad; se trata como información estructural.

In [1]:
# ============================================================
# 08_labs_diarios_DIANA.ipynb
# ------------------------------------------------------------
# Entrada:
#   - df_windows (en memoria) con:
#       icu_stay_id, subject_id, hadm_id, day_idx, window_start, window_end
#     (idealmente el mismo df_windows que usaste en 06)
#
# Salida:
#   - 08_labs_diarios.parquet
#
# Labs objetivo (mínimo):
#   - WBC
#   - Lactate
#
# Notas:
# - labevents.charttime es DATETIME en MIMIC-IV BigQuery
# - Ventanas en tabla temporal para join eficiente
# ============================================================

In [2]:
import pandas as pd
import numpy as np
from google.cloud import bigquery

In [3]:
# -----------------------------
# 0) Configuración
# -----------------------------
PROJECT_ID = "mimic-pruebas"
HOSP = "physionet-data.mimiciv_3_1_hosp"
df_windows = pd.read_parquet("05_ventanas_24h.parquet")

DATASET_TMP = "scratch"
TMP_TABLE_NAME = "tmp_08_windows"

client = bigquery.Client(project=PROJECT_ID)
tmp_table = f"{PROJECT_ID}.{DATASET_TMP}.{TMP_TABLE_NAME}"

print("PROJECT_ID:", PROJECT_ID)
print("HOSP:", HOSP)
print("tmp_table:", tmp_table)

PROJECT_ID: mimic-pruebas
HOSP: physionet-data.mimiciv_3_1_hosp
tmp_table: mimic-pruebas.scratch.tmp_08_windows


In [4]:
# -----------------------------
# 1) Validar df_windows
# -----------------------------
required_cols = {"icu_stay_id","subject_id","hadm_id","day_idx","window_start","window_end"}
missing = required_cols - set(df_windows.columns)
if missing:
    raise ValueError(f"Missing columns in df_windows: {missing}")

df_windows = df_windows.copy()
df_windows["icu_stay_id"] = df_windows["icu_stay_id"].astype("Int64")
df_windows["subject_id"]  = df_windows["subject_id"].astype("Int64")
df_windows["hadm_id"]     = df_windows["hadm_id"].astype("Int64")
df_windows["day_idx"]     = df_windows["day_idx"].astype(int)

df_windows["window_start"] = pd.to_datetime(df_windows["window_start"]).astype("datetime64[ns]")
df_windows["window_end"]   = pd.to_datetime(df_windows["window_end"]).astype("datetime64[ns]")

df_windows = df_windows.dropna(subset=["hadm_id"]).copy()
df_windows["hadm_id"] = df_windows["hadm_id"].astype("int64")

global_start = df_windows["window_start"].min()
global_end   = df_windows["window_end"].max()

print("Global window range:", global_start, "->", global_end)
print("N windows:", len(df_windows), "| N hadm:", df_windows["hadm_id"].nunique())

Global window range: 2110-01-13 20:00:00 -> 2214-07-27 16:00:00
N windows: 1130610 | N hadm: 18190


In [5]:
# -----------------------------
# 2) Subir ventanas a BigQuery
# -----------------------------
df_win_small = df_windows[["icu_stay_id","subject_id","hadm_id","day_idx","window_start","window_end"]].copy()

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
load_job = client.load_table_from_dataframe(df_win_small, tmp_table, job_config=job_config)
load_job.result()

print("Uploaded temp table:", tmp_table)
print(client.query(f"SELECT COUNT(*) AS n FROM `{tmp_table}`", location="US").to_dataframe())

Uploaded temp table: mimic-pruebas.scratch.tmp_08_windows


E0000 00:00:1769594966.453956  948769 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


         n
0  1130610


In [6]:
# -----------------------------
# 3) Descubrir itemids en d_labitems (NO asumir)
# -----------------------------
sql_labitems = f"""
SELECT itemid, label, fluid, category
FROM `{HOSP}.d_labitems`
WHERE
  REGEXP_CONTAINS(LOWER(label), r'\\bwbc\\b|white\\s+blood\\s+cell')
  OR REGEXP_CONTAINS(LOWER(label), r'\\blactate\\b')
ORDER BY itemid
"""
df_labitems = client.query(sql_labitems, location="US").to_dataframe()
display(df_labitems)

# Elegimos candidatos WBC y lactato por label (conservador)
wbc_candidates = df_labitems[df_labitems["label"].str.lower().str.contains("wbc|white blood", regex=True, na=False)].copy()
lac_candidates = df_labitems[df_labitems["label"].str.lower().str.contains("lactate", na=False)].copy()

print("\nWBC candidates:")
display(wbc_candidates)

print("\nLactate candidates:")
display(lac_candidates)

# Si hay muchos, nos quedamos con los itemid más plausibles por label
# (En práctica suele haber 1-3 relevantes)
WBC_ITEMIDS = wbc_candidates["itemid"].dropna().astype(int).unique().tolist()
LAC_ITEMIDS = lac_candidates["itemid"].dropna().astype(int).unique().tolist()

if len(WBC_ITEMIDS) == 0 or len(LAC_ITEMIDS) == 0:
    raise ValueError("No se han encontrado itemids para WBC o lactato. Revisa d_labitems.")

print("WBC_ITEMIDS:", WBC_ITEMIDS)
print("LAC_ITEMIDS:", LAC_ITEMIDS)

itemids_sql = ", ".join(map(str, sorted(set(WBC_ITEMIDS + LAC_ITEMIDS))))

E0000 00:00:1769594968.637644  948769 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,itemid,label,fluid,category
0,50813,Lactate,Blood,Blood Gas
1,50843,"Lactate Dehydrogenase, Ascites",Ascites,Chemistry
2,50954,Lactate Dehydrogenase (LD),Blood,Chemistry
3,51054,"Lactate Dehydrogenase, Pleural",Pleural,Chemistry
4,51300,WBC Count,Blood,Hematology
5,51301,White Blood Cells,Blood,Hematology
6,51516,WBC,Urine,Hematology
7,51517,WBC Casts,Urine,Hematology
8,51518,WBC Clumps,Urine,Hematology
9,51755,White Blood Cells,Blood,Chemistry



WBC candidates:


,itemid,label,fluid,category
4,51300,WBC Count,Blood,Hematology
5,51301,White Blood Cells,Blood,Hematology
6,51516,WBC,Urine,Hematology
7,51517,WBC Casts,Urine,Hematology
8,51518,WBC Clumps,Urine,Hematology
9,51755,White Blood Cells,Blood,Chemistry
10,51756,White Blood Cells,Blood,Chemistry
13,52407,WBC,Stool,Hematology



Lactate candidates:


,itemid,label,fluid,category
0,50813,Lactate,Blood,Blood Gas
1,50843,"Lactate Dehydrogenase, Ascites",Ascites,Chemistry
2,50954,Lactate Dehydrogenase (LD),Blood,Chemistry
3,51054,"Lactate Dehydrogenase, Pleural",Pleural,Chemistry
11,51795,"Lactate Dehydrogenase, CSF",Cerebrospinal Fluid,Chemistry
12,51944,"Lactate Dehydrogenase, Stool",Stool,Chemistry
14,52442,Lactate,Blood,Blood Gas
15,53154,Lactate,Blood,Chemistry


WBC_ITEMIDS: [51300, 51301, 51516, 51517, 51518, 51755, 51756, 52407]
LAC_ITEMIDS: [50813, 50843, 50954, 51054, 51795, 51944, 52442, 53154]


In [7]:
# -----------------------------
# 4) Query labevents dentro de ventanas
# -----------------------------
# charttime es DATETIME. window_start/end son DATETIME (tabla subida).
sql_labs = f"""
WITH w AS (
  SELECT
    subject_id,
    hadm_id,
    icu_stay_id,
    day_idx,
    window_start,
    window_end
  FROM `{tmp_table}`
),
le AS (
  SELECT
    le.hadm_id,
    le.charttime,          -- DATETIME
    le.itemid,
    le.valuenum,
    le.valueuom
  FROM `{HOSP}.labevents` le
  WHERE le.itemid IN ({itemids_sql})
    AND le.valuenum IS NOT NULL
    AND le.charttime IS NOT NULL
    AND le.charttime >= DATETIME('{global_start}')
    AND le.charttime <  DATETIME('{global_end}')
)
SELECT
  w.subject_id,
  w.hadm_id,
  w.icu_stay_id,
  w.day_idx,
  le.itemid,
  le.valuenum,
  le.valueuom
FROM w
JOIN le
  ON le.hadm_id = w.hadm_id
 AND le.charttime >= w.window_start
 AND le.charttime <  w.window_end
"""
df_labs_long = client.query(sql_labs, location="US").to_dataframe()
print("df_labs_long shape:", df_labs_long.shape)
display(df_labs_long.head())

E0000 00:00:1769594974.873886  948769 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


df_labs_long shape: (2920842, 7)


,subject_id,hadm_id,icu_stay_id,day_idx,itemid,valuenum,valueuom
0,11130556,28389443,31700014,0,51516,0.0,#/hpf
1,11135588,22848293,34331323,16,51516,0.0,#/hpf
2,11135588,22848293,31618350,16,51516,0.0,#/hpf
3,11135588,22848293,34331323,16,51516,0.0,#/hpf
4,11135588,22848293,31618350,16,51516,0.0,#/hpf


In [8]:
# -----------------------------
# 5) Map itemid -> variable (WBC/Lactate)
# -----------------------------
df_labs_long["itemid"] = df_labs_long["itemid"].astype(int)

def map_lab(itemid: int) -> str:
    if itemid in WBC_ITEMIDS:
        return "WBC"
    if itemid in LAC_ITEMIDS:
        return "Lactate"
    return None

df_labs_long["lab"] = df_labs_long["itemid"].apply(map_lab)
df_labs_long = df_labs_long[df_labs_long["lab"].notna()].copy()


In [9]:
# -----------------------------
# 6) Normalizaciones y filtros plausibles
# -----------------------------
# Lactate: si viene en mg/dL, convertir a mmol/L
# lactate MW ~ 90.08 g/mol => 1 mmol/L = 9.008 mg/dL
# mmol/L = mg/dL / 9.008
def normalize_lactate(value, uom):
    if pd.isna(value):
        return np.nan
    v = float(value)
    u = str(uom).strip().lower() if uom is not None else ""
    if u in ["mmol/l", "mmol/liter", "mmol/lit", "mmol/l."]:
        return v
    if u in ["mg/dl", "mg/dl."]:
        return v / 9.008
    # si no sabemos la unidad, no convertimos (pero lo dejamos para inspección)
    return v

# WBC: normalmente K/uL o 10^3/uL; no convertimos, solo filtramos plausibilidad.
def plausibility_filter(row):
    lab = row["lab"]
    v = row["valuenum"]
    if pd.isna(v):
        return np.nan
    v = float(v)
    if lab == "WBC":
        # rango amplio (evita outliers absurdos)
        return v if (0.1 <= v <= 200) else np.nan
    if lab == "Lactate":
        v2 = normalize_lactate(v, row.get("valueuom", ""))
        return v2 if (0.1 <= v2 <= 30) else np.nan
    return np.nan

df_labs_long["value_norm"] = df_labs_long.apply(plausibility_filter, axis=1)

# QA unidades
print("\nTop units per lab:")
display(df_labs_long.groupby("lab")["valueuom"].value_counts().head(20))


Top units per lab:


lab      valueuom
Lactate  mmol/L       894743
         IU/L         336375
WBC      K/uL        1594059
         #/hpf         95309
         #/lpf           356
Name: count, dtype: int64

In [10]:
# -----------------------------
# 7) Agregar por día (mediana) y pivot
# -----------------------------
df_labs_agg = (
    df_labs_long
    .dropna(subset=["value_norm"])
    .groupby(["subject_id","hadm_id","icu_stay_id","day_idx","lab"], as_index=False)["value_norm"]
    .median()
)

df_labs_daily = (
    df_labs_agg
    .pivot_table(
        index=["subject_id","hadm_id","icu_stay_id","day_idx"],
        columns="lab",
        values="value_norm",
        aggfunc="first"
    )
    .reset_index()
    .sort_values(["subject_id","hadm_id","icu_stay_id","day_idx"])
    .reset_index(drop=True)
)

print("\nDaily labs shape:", df_labs_daily.shape)
display(df_labs_daily.head(10))


Daily labs shape: (170806, 6)


lab,subject_id,hadm_id,icu_stay_id,day_idx,Lactate,WBC
0,10001217,24597018,37067082,0,NaN,17.35
1,10001217,24597018,37067082,1,NaN,14.80
2,10002428,20321825,34807493,0,0.90,3.80
3,10002428,20321825,34807493,1,NaN,4.90
4,10002428,23473524,35479615,0,0.90,5.50
5,10002428,23473524,35479615,1,NaN,5.10
6,10002428,28662225,33987268,0,2.40,33.40
7,10002428,28662225,33987268,1,1.90,28.60
8,10002428,28662225,33987268,2,2.25,17.85
9,10002428,28662225,33987268,3,1.80,20.40


In [11]:
# -----------------------------
# 8) Dominios DIANA (labs)
# -----------------------------
# Lactate domain
df_labs_daily["lactate_measured"] = df_labs_daily["Lactate"].notna().astype(int)
df_labs_daily["lactate_low"] = np.where(
    df_labs_daily["Lactate"].notna(),
    (df_labs_daily["Lactate"] < 2.0).astype(int),
    np.nan
)

# WBC domains
WBC_LOW, WBC_HIGH = 4.0, 12.0

df_labs_daily["wbc_measured"] = df_labs_daily["WBC"].notna().astype(int)
df_labs_daily["wbc_in_range"] = np.where(
    df_labs_daily["WBC"].notna(),
    ((df_labs_daily["WBC"] >= WBC_LOW) & (df_labs_daily["WBC"] <= WBC_HIGH)).astype(int),
    np.nan
)

# WBC trend to normality (per stay)
# distance to [4,12]; trend = +1 si se acerca (distance baja), -1 si se aleja, 0 si igual.
def wbc_trend(series: pd.Series) -> pd.Series:
    s = series.copy()
    dist = (s - s.clip(WBC_LOW, WBC_HIGH)).abs()
    d = dist.diff()
    # d < 0 => se acerca => +1; d > 0 => empeora => -1; 0 => 0
    out = d.apply(lambda x: (1 if x < 0 else (-1 if x > 0 else 0)) if pd.notna(x) else np.nan)
    return out

df_labs_daily["wbc_trend"] = (
    df_labs_daily
    .groupby(["subject_id","hadm_id","icu_stay_id"])["WBC"]
    .apply(wbc_trend)
    .reset_index(level=[0,1,2], drop=True)
)

df_labs_daily["wbc_improving"] = np.where(
    df_labs_daily["wbc_trend"].notna(),
    (df_labs_daily["wbc_trend"] == 1).astype(int),
    np.nan
)

In [12]:
# -----------------------------
# 9) QA final
# -----------------------------
for c in ["WBC","Lactate","lactate_low","wbc_in_range","wbc_trend"]:
    if c in df_labs_daily.columns:
        print(f"{c} NaN rate:", df_labs_daily[c].isna().mean())

print("\nSummary WBC:")
display(df_labs_daily["WBC"].describe(percentiles=[.01,.05,.5,.95,.99]))

print("\nSummary Lactate (mmol/L if converted):")
display(df_labs_daily["Lactate"].describe(percentiles=[.01,.05,.5,.95,.99]))

WBC NaN rate: 0.00502324274322916
Lactate NaN rate: 0.6674941161317518
lactate_low NaN rate: 0.6674941161317518
wbc_in_range NaN rate: 0.00502324274322916
wbc_trend NaN rate: 0.12753064880624804

Summary WBC:


count    169948.000000
mean         12.714204
std           8.523973
min           0.100000
1%            0.500000
5%            3.700000
50%          11.000000
95%          27.000000
99%          43.600000
max         196.700000
Name: WBC, dtype: float64


Summary Lactate (mmol/L if converted):


count    56794.000000
mean         1.960696
std          1.847467
min          0.200000
1%           0.600000
5%           0.700000
50%          1.500000
95%          4.600000
99%         10.500000
max         30.000000
Name: Lactate, dtype: float64

In [13]:
# -----------------------------
# 10) Guardar
# -----------------------------
out_path = "08_labs_diarios.parquet"
df_labs_daily.to_parquet(out_path, index=False)
print("Saved:", out_path)

df_labs_daily.head(10)

Saved: 08_labs_diarios.parquet


lab,subject_id,hadm_id,icu_stay_id,day_idx,Lactate,WBC,lactate_measured,lactate_low,wbc_measured,wbc_in_range,wbc_trend,wbc_improving
0,10001217,24597018,37067082,0,NaN,17.35,0,NaN,1,0.0,NaN,NaN
1,10001217,24597018,37067082,1,NaN,14.80,0,NaN,1,0.0,1.0,1.0
2,10002428,20321825,34807493,0,0.90,3.80,1,1.0,1,0.0,NaN,NaN
3,10002428,20321825,34807493,1,NaN,4.90,0,NaN,1,1.0,1.0,1.0
4,10002428,23473524,35479615,0,0.90,5.50,1,1.0,1,1.0,NaN,NaN
5,10002428,23473524,35479615,1,NaN,5.10,0,NaN,1,1.0,0.0,0.0
6,10002428,28662225,33987268,0,2.40,33.40,1,0.0,1,0.0,NaN,NaN
7,10002428,28662225,33987268,1,1.90,28.60,1,1.0,1,0.0,1.0,1.0
8,10002428,28662225,33987268,2,2.25,17.85,1,0.0,1,0.0,1.0,1.0
9,10002428,28662225,33987268,3,1.80,20.40,1,1.0,1,0.0,-1.0,0.0
